# Data Visualization in Python
## Matplotlib · Seaborn · Plotly

---

## Table of Contents

**Matplotlib**
1. Figures, Axes, and the Object Model
2. Line, Scatter, and Bar Charts
3. Histograms, Box Plots, and Distributions
4. Subplots, Layouts, and Styling
5. Annotations, Text, and Custom Formatting

**Seaborn**
6. Statistical Plots — relplot, displot, catplot
7. Heatmaps, Pair Plots, and Joint Plots
8. Regression and FacetGrid

**Plotly**
9. Interactive Plots with Plotly Express
10. Plotly Graph Objects and Dashboards

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
    print('Seaborn ready.')
except ImportError:
    print('Install: pip install seaborn')

try:
    import plotly.express as px
    import plotly.graph_objects as go
    print('Plotly ready.')
except ImportError:
    print('Install: pip install plotly')

np.random.seed(42)

# Shared sample data used throughout
n = 200
rng = np.random.default_rng(42)

df = pd.DataFrame({
    'department': rng.choice(['Engineering','Marketing','Sales','HR','Finance'], size=n),
    'salary':     rng.integers(45000, 140000, size=n),
    'tenure':     rng.integers(1, 20, size=n),
    'score':      rng.normal(75, 15, size=n).clip(0, 100),
    'gender':     rng.choice(['M','F'], size=n),
    'promoted':   rng.choice([True, False], size=n, p=[0.3, 0.7])
})

print('Dataset ready:', df.shape)
print(df.head())

# Section 1 — Figures, Axes, and the Object Model

## Concept

Matplotlib has two APIs:
- **pyplot API** (`plt.plot(...)`) — quick, stateful, MATLAB-style.
- **Object API** (`fig, ax = plt.subplots()`) — explicit, composable, preferred for production.

Always use the object API for anything beyond a quick prototype.

## Technical Deep Dive

**Hierarchy:**
```
Figure
  └── Axes (one or many)
        ├── XAxis / YAxis
        ├── Tick labels
        ├── Title
        └── Lines, Patches, Text, Collections
```

| Method | Description |
|---|---|
| `fig, ax = plt.subplots(figsize=(w, h))` | Create figure + axes |
| `ax.set_title()` | Axes title |
| `ax.set_xlabel() / set_ylabel()` | Axis labels |
| `ax.set_xlim() / set_ylim()` | Axis limits |
| `ax.legend()` | Show legend |
| `ax.grid(True)` | Enable grid |
| `fig.savefig('out.png', dpi=150)` | Export |
| `plt.tight_layout()` | Fix overlapping labels |


In [ ]:
# Object API — the correct way
fig, ax = plt.subplots(figsize=(8, 4))

x = np.linspace(0, 4 * np.pi, 300)
ax.plot(x, np.sin(x), label='sin(x)', linewidth=2, color='steelblue')
ax.plot(x, np.cos(x), label='cos(x)', linewidth=2, color='tomato', linestyle='--')

ax.set_title('Sine and Cosine', fontsize=14, fontweight='bold')
ax.set_xlabel('x (radians)')
ax.set_ylabel('Amplitude')
ax.set_xlim(0, 4*np.pi)
ax.set_ylim(-1.4, 1.4)
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

In [ ]:
# Matplotlib styles
print('Available styles:', plt.style.available[:10])

with plt.style.context('seaborn-v0_8-darkgrid'):
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(x, np.sin(x), color='gold', linewidth=2)
    ax.set_title('seaborn-darkgrid style')
    plt.tight_layout()
    plt.show()

## Summary

- Always use `fig, ax = plt.subplots()` — avoid stateful pyplot in production.
- Every visual element is an artist object; configure it through the `ax` and `fig` references.
- Save with `fig.savefig('name.png', dpi=150, bbox_inches='tight')`.

---


# Section 2 — Line, Scatter, and Bar Charts

## Concept

The three most common chart types:
- **Line** — trends over a continuous axis (time series, functions).
- **Scatter** — relationship between two numeric variables.
- **Bar** — comparing categories.


In [ ]:
# Line chart — time series
dates = pd.date_range('2023-01-01', periods=52, freq='W')
weekly_sales = rng.normal(10000, 1500, size=52).cumsum() + 50000

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates, weekly_sales, linewidth=1.5, color='steelblue', label='Sales')
ax.fill_between(dates, weekly_sales, alpha=0.15, color='steelblue')

ax.set_title('Weekly Sales 2023', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Sales ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
ax.grid(True, axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot — salary vs tenure, colored by department
fig, ax = plt.subplots(figsize=(8, 5))

departments = df['department'].unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(departments)))

for dept, color in zip(departments, colors):
    mask = df['department'] == dept
    ax.scatter(df.loc[mask, 'tenure'], df.loc[mask, 'salary'],
               label=dept, color=color, alpha=0.7, s=50, edgecolors='white', linewidths=0.5)

ax.set_title('Salary vs Tenure by Department')
ax.set_xlabel('Tenure (years)')
ax.set_ylabel('Salary ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
ax.legend(title='Department', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart — average salary per department
dept_stats = df.groupby('department')['salary'].agg(['mean','std']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(dept_stats.index, dept_stats['mean'],
              yerr=dept_stats['std'], capsize=5,
              color='steelblue', alpha=0.8, edgecolor='white')

# Add value labels on bars
for bar, val in zip(bars, dept_stats['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
            f'${val/1000:.0f}k', ha='center', va='bottom', fontsize=9)

ax.set_title('Average Salary by Department (± 1 std)')
ax.set_ylabel('Salary ($)')
ax.set_ylim(0, 170000)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

1. Create a horizontal bar chart of headcount per department (use `ax.barh`).
2. Plot salary vs score as a scatter, with point size proportional to tenure.
3. Add a trend line (regression line) to a scatter plot using `np.polyfit`.
4. Create a grouped bar chart: salary mean for M vs F per department.

## Mini Challenge

Create a dual-axis chart: bar chart of headcount per department (left y-axis)
overlaid with a line showing average salary (right y-axis). Use `ax.twinx()`.

## Best Practices

- Sort bars for bar charts — unsorted bars are hard to compare.
- Add error bars to show uncertainty (std, CI).
- Use `alpha` on scatter for overplotting.
- Avoid 3D bar charts — they distort perception.

## Summary

- Line: use `fill_between` for area emphasis.
- Scatter: encode extra dimensions via `color`, `size`, `alpha`.
- Bar: sort, add value labels, include error bars for statistical rigor.

---


### Exercise and Challenge Solutions — Section 2


In [ ]:
# Exercise 1: horizontal bar
# YOUR CODE HERE

In [ ]:
# Exercise 3: scatter with trend line
# YOUR CODE HERE

In [ ]:
# Mini Challenge: dual-axis chart
# YOUR CODE HERE

# Section 3 — Histograms, Box Plots, and Distributions

## Concept

Distribution charts reveal the shape, spread, and outliers of a variable.
Use these before running any statistical model.


In [ ]:
# Histogram with KDE overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw histogram
axes[0].hist(df['salary'], bins=25, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Salary Distribution (Histogram)')
axes[0].set_xlabel('Salary ($)')
axes[0].set_ylabel('Count')

# Stacked histograms by gender
for gender, color in zip(['M', 'F'], ['steelblue', 'tomato']):
    subset = df[df['gender'] == gender]['salary']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=gender, edgecolor='white')
axes[1].set_title('Salary by Gender (Overlapping Histograms)')
axes[1].set_xlabel('Salary ($)')
axes[1].legend(title='Gender')

plt.tight_layout()
plt.show()

In [ ]:
# Box plots — salary per department
dept_order = df.groupby('department')['salary'].median().sort_values(ascending=False).index
dept_data  = [df[df['department'] == d]['salary'].values for d in dept_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
bp = axes[0].boxplot(dept_data, labels=dept_order, patch_artist=True, notch=True)
colors_bp = plt.cm.Set2(np.linspace(0, 1, len(dept_order)))
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
axes[0].set_title('Salary Distribution — Box Plot')
axes[0].set_ylabel('Salary ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))

# Violin plot
vp = axes[1].violinplot(dept_data, showmedians=True, showextrema=True)
axes[1].set_xticks(range(1, len(dept_order)+1))
axes[1].set_xticklabels(dept_order, rotation=15)
axes[1].set_title('Salary Distribution — Violin Plot')
axes[1].set_ylabel('Salary ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))

plt.tight_layout()
plt.show()

## Exercises

1. Plot a cumulative histogram (ECDF) of salary.
2. Create a box plot comparing score distributions across departments, with jittered points overlaid.
3. Plot salary with a log-scale x-axis — does the distribution look more normal?

## Summary

- Histogram: see shape and bin counts.
- Box plot: IQR, median, outliers — good for comparing groups.
- Violin: more information about distribution shape than a box plot.
- Always check distributions before modeling — normality assumptions matter.

---


### Exercise and Challenge Solutions — Section 3


In [ ]:
# ECDF
sorted_sal = np.sort(df['salary'])
ecdf = np.arange(1, len(sorted_sal)+1) / len(sorted_sal)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sorted_sal, ecdf, linewidth=2, color='steelblue')
ax.axhline(0.5, color='gray', linestyle='--', label='Median')
ax.set_xlabel('Salary ($)'); ax.set_ylabel('Cumulative Proportion')
ax.set_title('Empirical CDF of Salary')
ax.legend(); plt.tight_layout(); plt.show()

# Section 4 — Subplots, Layouts, and Styling

## Concept

Dashboards often need multiple plots in one figure.
`plt.subplots`, `GridSpec`, and `subplot_mosaic` give full layout control.


In [ ]:
# 2x2 subplot grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Employee Data Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Top-left: salary histogram
axes[0,0].hist(df['salary'], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
axes[0,0].set_title('Salary Distribution')
axes[0,0].set_xlabel('Salary ($)')

# Top-right: department pie
hc = df['department'].value_counts()
axes[0,1].pie(hc.values, labels=hc.index, autopct='%1.1f%%', startangle=90,
              colors=plt.cm.Set3.colors[:len(hc)])
axes[0,1].set_title('Department Distribution')

# Bottom-left: scatter salary vs score
axes[1,0].scatter(df['score'], df['salary'], alpha=0.4, s=25, c=df['tenure'], cmap='viridis')
axes[1,0].set_title('Salary vs Score (color=tenure)')
axes[1,0].set_xlabel('Score'); axes[1,0].set_ylabel('Salary')

# Bottom-right: avg salary bar
avg = df.groupby('department')['salary'].mean().sort_values(ascending=False)
axes[1,1].bar(avg.index, avg.values, color=plt.cm.Set2.colors[:5], alpha=0.85)
axes[1,1].set_title('Avg Salary by Department')
axes[1,1].set_ylabel('Salary ($)')
axes[1,1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# subplot_mosaic — non-uniform layouts
layout = [
    ['big', 'big', 'top_right'],
    ['bot_left', 'bot_mid', 'top_right']
]
fig, axd = plt.subplot_mosaic(layout, figsize=(12, 7))

# Big panel
axd['big'].scatter(df['tenure'], df['salary'], c=df['score'], cmap='coolwarm', alpha=0.6, s=50)
axd['big'].set_title('Salary vs Tenure (color=score)')
axd['big'].set_xlabel('Tenure'); axd['big'].set_ylabel('Salary')

# Top right
axd['top_right'].hist(df['score'], bins=15, color='orchid', alpha=0.8, edgecolor='white')
axd['top_right'].set_title('Score Distribution')

# Bottom left
axd['bot_left'].bar(['Promoted','Not'], [df['promoted'].sum(), (~df['promoted']).sum()], color=['teal','lightcoral'])
axd['bot_left'].set_title('Promotion Status')

# Bottom mid
axd['bot_mid'].boxplot([df[df['gender']=='M']['salary'], df[df['gender']=='F']['salary']], labels=['M','F'], patch_artist=True)
axd['bot_mid'].set_title('Salary by Gender')

fig.suptitle('Mosaic Layout Dashboard', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## Summary

- `plt.subplots(rows, cols)` for uniform grids.
- `plt.subplot_mosaic(layout)` for non-uniform layouts — very expressive.
- `fig.suptitle` for overall figure title.
- `plt.tight_layout()` / `fig.tight_layout()` fixes padding.

---


# Section 5 — Annotations, Text, and Custom Formatting

## Concept

Annotations communicate insights directly on the plot.
Custom formatters handle currency, percentages, and scientific notation.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(0, 4*np.pi, 500)
y = np.sin(x) * np.exp(-0.1 * x)
ax.plot(x, y, color='steelblue', linewidth=2)

# Max point annotation
idx_max = np.argmax(y)
ax.annotate(
    f'Max: ({x[idx_max]:.2f}, {y[idx_max]:.2f})',
    xy=(x[idx_max], y[idx_max]),
    xytext=(x[idx_max]+0.8, y[idx_max]+0.1),
    arrowprops=dict(arrowstyle='->', color='tomato', lw=1.5),
    fontsize=10, color='tomato'
)

# Shaded region
ax.axvspan(2*np.pi, 3*np.pi, alpha=0.1, color='green', label='region of interest')
ax.text(2.5*np.pi, 0.3, 'region', ha='center', fontsize=9, color='green')

# Horizontal reference line
ax.axhline(0, color='black', linewidth=0.8)

ax.set_title('Damped Sine Wave with Annotations', fontsize=13)
ax.set_xlabel('x'); ax.set_ylabel('Amplitude')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter, PercentFormatter, LogLocator

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Currency formatter
axes[0].bar(range(5), [50000, 75000, 90000, 120000, 60000], color='steelblue', alpha=0.8)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'${v/1000:.0f}k'))
axes[0].set_title('Currency Formatter')

# Percent formatter
axes[1].plot([0, 0.1, 0.3, 0.6, 0.85, 0.95, 1.0])
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[1].set_title('Percent Formatter')

# Log scale
values = [1, 10, 100, 1000, 10000, 100000]
axes[2].semilogy(range(len(values)), values, 'o-', color='tomato')
axes[2].set_title('Log Scale')
axes[2].set_ylabel('Value (log)')

plt.tight_layout(); plt.show()

## Summary

- `ax.annotate` draws arrows + text to highlight specific data points.
- `ax.axvspan` / `ax.axhspan` shade regions.
- `FuncFormatter`, `PercentFormatter` make axes human-readable.
- Use `ax.set_yscale('log')` for data spanning multiple orders of magnitude.

---

# SEABORN

---


# Section 6 — Seaborn Statistical Plots

## Concept

Seaborn is built on top of Matplotlib and provides high-level, statistically-aware chart functions.
It works directly with DataFrames and handles grouping, CI, and legends automatically.

## Technical Deep Dive

| Function | Use Case |
|---|---|
| `sns.scatterplot` | Scatter with hue/size/style |
| `sns.lineplot` | Line with CI band |
| `sns.barplot` | Bar with CI |
| `sns.boxplot` | Box plots |
| `sns.violinplot` | Violin plots |
| `sns.histplot` | Histogram with KDE |
| `sns.kdeplot` | Kernel density estimate |
| `sns.stripplot` | Jittered points |
| `sns.swarmplot` | Non-overlapping points |
| `sns.countplot` | Bar chart of counts |

**Figure-level functions** (`relplot`, `displot`, `catplot`) automatically handle `FacetGrid`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter with hue + size
sns.scatterplot(
    data=df, x='tenure', y='salary', hue='department', size='score',
    sizes=(30, 200), alpha=0.7, palette='Set2', ax=axes[0]
)
axes[0].set_title('Scatter: Salary vs Tenure')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v/1000:.0f}k'))
axes[0].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

# Bar with CI
sns.barplot(
    data=df, x='department', y='salary', hue='gender',
    palette=['steelblue','tomato'], errorbar='ci', capsize=0.1, ax=axes[1]
)
axes[1].set_title('Avg Salary by Dept & Gender (95% CI)')
axes[1].set_ylabel('Salary ($)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v/1000:.0f}k'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram + KDE
sns.histplot(data=df, x='salary', hue='gender', kde=True, bins=20,
             palette=['steelblue','tomato'], alpha=0.6, ax=axes[0])
axes[0].set_title('Salary Distribution by Gender')

# KDE by department
for dept in df['department'].unique():
    sns.kdeplot(data=df[df['department']==dept], x='salary', label=dept, ax=axes[1])
axes[1].set_title('KDE: Salary per Department')
axes[1].legend(fontsize=8)

# Violin + strip overlay
sns.violinplot(data=df, x='department', y='salary', palette='Set3', inner=None, ax=axes[2])
sns.stripplot(data=df, x='department', y='salary', color='black', alpha=0.2, size=3, ax=axes[2])
axes[2].set_title('Salary Violin + Jitter')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.show()

In [ ]:
# catplot — figure-level, auto FacetGrid
g = sns.catplot(
    data=df, kind='box',
    x='department', y='salary', hue='promoted',
    palette='Set1', height=4, aspect=2
)
g.set_axis_labels('Department', 'Salary ($)')
g.fig.suptitle('Salary by Dept and Promotion Status', y=1.02)
plt.tight_layout(); plt.show()

## Exercises

1. Create a `countplot` of `department`, colored by `gender`.
2. Plot `sns.boxplot` of score per department, sorted by median.
3. Use `relplot` to show salary vs tenure, with different plot panels per gender.
4. Create a `swarmplot` of salary per department.

## Summary

- Seaborn handles grouping, CI, and KDE automatically.
- Figure-level functions (`catplot`, `relplot`, `displot`) support `col` / `row` faceting.
- Combine `violinplot` + `stripplot` for density + individual points.

---


### Exercise and Challenge Solutions — Section 6


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=df, x='department', hue='gender', palette=['steelblue','tomato'], ax=axes[0])
axes[0].set_title('Headcount by Dept & Gender'); axes[0].tick_params(axis='x', rotation=20)

dept_order_med = df.groupby('department')['score'].median().sort_values(ascending=False).index.tolist()
sns.boxplot(data=df, x='department', y='score', order=dept_order_med, palette='Set2', ax=axes[1])
axes[1].set_title('Score by Department (sorted by median)'); axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
g = sns.relplot(data=df, x='tenure', y='salary', col='gender', hue='department',
                kind='scatter', alpha=0.6, height=4, aspect=1.2, palette='Set2')
g.fig.suptitle('Salary vs Tenure (faceted by Gender)', y=1.02)
plt.tight_layout(); plt.show()

# Section 7 — Heatmaps, Pair Plots, and Joint Plots

## Concept

- **Heatmap** — visualize 2D matrices (correlation, pivot tables, confusion matrices).
- **Pair plot** — scatter matrix of all numeric columns, colored by group.
- **Joint plot** — bivariate distribution with marginal distributions.


In [ ]:
# Correlation heatmap
corr = df[['salary','tenure','score']].corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, ax=axes[0])
axes[0].set_title('Correlation Matrix')

# Pivot table heatmap — avg salary by dept x gender
pivot = df.pivot_table(values='salary', index='department', columns='gender', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=axes[1])
axes[1].set_title('Avg Salary: Department x Gender')

plt.tight_layout(); plt.show()

In [ ]:
# Pair plot
g = sns.pairplot(
    df[['salary','tenure','score','department']],
    hue='department', palette='Set2',
    diag_kind='kde', plot_kws={'alpha': 0.5, 's': 30}
)
g.fig.suptitle('Pair Plot — Numeric Variables by Department', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Joint plot — salary vs score
g = sns.jointplot(
    data=df, x='score', y='salary',
    kind='scatter', hue='gender',
    palette=['steelblue','tomato'],
    alpha=0.6, height=6
)
g.fig.suptitle('Salary vs Score Joint Distribution', y=1.01)
plt.tight_layout(); plt.show()

## Summary

- `sns.heatmap` with `annot=True` is the standard correlation visualization.
- `pairplot` gives a complete overview of all pairwise relationships.
- `jointplot` focuses on two variables with marginal distributions.

---


# Section 8 — Regression Plots and FacetGrid

## Concept

- `sns.lmplot` / `sns.regplot` — scatter + regression line with CI.
- `sns.residplot` — residual plot for regression diagnostics.
- `FacetGrid` — multipanel plots from a DataFrame.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# regplot — single regression
sns.regplot(data=df, x='tenure', y='salary', scatter_kws={'alpha':0.5, 's':30},
            line_kws={'color':'tomato', 'lw':2}, ci=95, ax=axes[0])
axes[0].set_title('Salary vs Tenure (Linear Regression)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v/1000:.0f}k'))

# lmplot per department (figure-level)
plt.close()  # close the current fig before lmplot creates its own
g = sns.lmplot(data=df, x='tenure', y='salary', hue='department',
               palette='Set2', scatter_kws={'alpha':0.4, 's':25},
               height=5, aspect=1.4)
g.fig.suptitle('Regression by Department', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# FacetGrid — histogram per department
g = sns.FacetGrid(df, col='department', col_wrap=3, height=3, sharey=False)
g.map(sns.histplot, 'salary', bins=12, color='steelblue', alpha=0.8)
g.set_axis_labels('Salary ($)', 'Count')
g.fig.suptitle('Salary Distribution per Department', y=1.02)
plt.tight_layout(); plt.show()

## Summary

- `regplot` adds regression with confidence interval to a scatter.
- `lmplot` is the figure-level version supporting `hue`, `col`, `row`.
- `FacetGrid` gives complete control over multipanel plots.

---

# PLOTLY — Interactive Visualization

---


# Section 9 — Interactive Plots with Plotly Express

## Concept

Plotly creates interactive HTML charts — hover, zoom, filter, and click are built-in.
**Plotly Express** (`px`) is a high-level API that mirrors seaborn's simplicity.
Ideal for notebooks, dashboards (Dash), and web reports.

## Technical Deep Dive

| `px` function | Chart type |
|---|---|
| `px.scatter` | Scatter |
| `px.line` | Line |
| `px.bar` | Bar |
| `px.histogram` | Histogram |
| `px.box` | Box plot |
| `px.violin` | Violin |
| `px.density_heatmap` | 2D histogram |
| `px.treemap` | Hierarchical treemap |
| `px.sunburst` | Sunburst chart |
| `px.choropleth` | Map |
| `px.scatter_3d` | 3D scatter |


In [ ]:
try:
    import plotly.express as px

    fig = px.scatter(
        df, x='tenure', y='salary',
        color='department', size='score',
        hover_data=['gender', 'promoted'],
        title='Interactive: Salary vs Tenure',
        template='plotly_white',
        labels={'tenure': 'Tenure (years)', 'salary': 'Salary ($)'}
    )
    fig.update_traces(marker=dict(opacity=0.7, line=dict(width=0.5, color='white')))
    fig.show()
except ImportError:
    print('Plotly not installed. Run: pip install plotly')

In [ ]:
try:
    # Animated bar chart
    dept_gender = df.groupby(['department','gender'])['salary'].mean().reset_index()
    fig = px.bar(
        dept_gender, x='department', y='salary', color='gender',
        barmode='group', title='Avg Salary by Department and Gender',
        template='plotly_white',
        color_discrete_map={'M': 'steelblue', 'F': 'tomato'}
    )
    fig.update_yaxes(tickprefix='$', tickformat='.0f')
    fig.show()

    # Box plot
    fig2 = px.box(
        df, x='department', y='salary', color='promoted',
        points='outliers', title='Salary Distribution by Dept + Promotion',
        template='plotly_white'
    )
    fig2.show()
except ImportError:
    print('Plotly not installed.')

In [ ]:
try:
    agg = df.groupby(['department','gender']).agg(
        headcount=('salary','count'),
        avg_salary=('salary','mean')
    ).reset_index()

    fig = px.treemap(
        agg, path=['department','gender'],
        values='headcount', color='avg_salary',
        color_continuous_scale='RdYlGn',
        title='Treemap: Headcount (size) and Avg Salary (color)',
        template='plotly_white'
    )
    fig.show()
except ImportError:
    print('Plotly not installed.')

In [ ]:
try:
    # Interactive time series with range slider
    dates = pd.date_range('2022-01-01', periods=365, freq='D')
    ts_df = pd.DataFrame({
        'date': dates,
        'value': rng.normal(100, 10, 365).cumsum() + 5000
    })

    fig = px.line(
        ts_df, x='date', y='value',
        title='Time Series with Range Slider',
        template='plotly_white'
    )
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=[
                    dict(count=1, label='1M', step='month'),
                    dict(count=3, label='3M', step='month'),
                    dict(step='all')
                ]
            ),
            rangeslider=dict(visible=True)
        )
    )
    fig.show()
except ImportError:
    print('Plotly not installed.')

## Exercises

1. Create an interactive scatter with `px.scatter_3d`: x=tenure, y=salary, z=score, color=department.
2. Create a `px.histogram` of salary faceted by department (`facet_col='department'`).
3. Create a `px.sunburst` of department → gender → promoted hierarchy.
4. Create a `px.violin` of salary by department with `box=True` to overlay a box plot.

## Summary

- Plotly Express generates interactive charts with one line.
- `hover_data`, `color`, `size`, `facet_col` are the main customization params.
- Range sliders, selectors, and dropdowns are built-in for time series.

---


### Exercise and Challenge Solutions — Section 9


In [ ]:
try:
    import plotly.express as px

    # 3D scatter
    px.scatter_3d(df, x='tenure', y='salary', z='score', color='department',
                  opacity=0.7, title='3D: Tenure / Salary / Score').show()

    # Histogram faceted
    px.histogram(df, x='salary', facet_col='department', nbins=15,
                 title='Salary Histograms per Department', template='plotly_white').show()

    # Sunburst
    agg2 = df.groupby(['department','gender','promoted']).size().reset_index(name='count')
    agg2['promoted'] = agg2['promoted'].map({True: 'Promoted', False: 'Not Promoted'})
    px.sunburst(agg2, path=['department','gender','promoted'], values='count',
                title='Sunburst: Dept > Gender > Promoted').show()

    # Violin with box
    px.violin(df, x='department', y='salary', box=True, color='department',
              title='Salary Violin + Box per Department', template='plotly_white').show()
except ImportError:
    print('Plotly not installed.')

# Section 10 — Plotly Graph Objects and Subplots

## Concept

**Plotly Graph Objects** (`go`) give full control:
custom traces, multiple y-axes, subplots, animations, and Dash integration.

## Technical Deep Dive

```python
from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=2)
fig.add_trace(go.Scatter(...), row=1, col=1)
fig.add_trace(go.Bar(...),     row=1, col=2)
fig.update_layout(title='...', template='plotly_white')
fig.show()
```


In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Salary Distribution','Dept Headcount','Salary vs Tenure','Score KDE'),
        vertical_spacing=0.12
    )

    # (1,1) Histogram
    fig.add_trace(go.Histogram(x=df['salary'], nbinsx=25, name='Salary',
                               marker_color='steelblue', opacity=0.8), row=1, col=1)

    # (1,2) Bar — headcount
    hc = df['department'].value_counts()
    fig.add_trace(go.Bar(x=hc.index, y=hc.values, name='Headcount',
                         marker_color='teal', opacity=0.85), row=1, col=2)

    # (2,1) Scatter
    fig.add_trace(go.Scatter(x=df['tenure'], y=df['salary'], mode='markers',
                             marker=dict(color=df['score'], colorscale='Viridis',
                                         size=5, opacity=0.6, showscale=True),
                             name='Salary vs Tenure'), row=2, col=1)

    # (2,2) KDE via histogram with histnorm
    fig.add_trace(go.Histogram(x=df['score'], nbinsx=20, histnorm='probability density',
                               name='Score KDE', marker_color='orchid', opacity=0.8), row=2, col=2)

    fig.update_layout(
        height=650, title_text='Company Dashboard — Plotly Go',
        template='plotly_white', showlegend=False
    )
    fig.show()
except ImportError:
    print('Plotly not installed.')

In [ ]:
try:
    # Candlestick chart (financial data)
    ohlc_dates = pd.date_range('2024-01-01', periods=60, freq='B')
    price = 100.0
    opens, highs, lows, closes = [], [], [], []
    for _ in ohlc_dates:
        o = price
        c = o + rng.normal(0, 2)
        h = max(o, c) + abs(rng.normal(0, 1))
        l = min(o, c) - abs(rng.normal(0, 1))
        opens.append(o); highs.append(h); lows.append(l); closes.append(c)
        price = c

    fig = go.Figure(data=[
        go.Candlestick(x=ohlc_dates, open=opens, high=highs, low=lows, close=closes, name='OHLC')
    ])
    fig.update_layout(title='Candlestick Chart (Simulated)', template='plotly_dark',
                      xaxis_rangeslider_visible=True)
    fig.show()
except ImportError:
    print('Plotly not installed.')

## Exercises

1. Create a dual y-axis plot: bar (headcount) + line (avg salary) using `make_subplots(specs=[[{'secondary_y': True}]])`.
2. Build a fully labeled funnel chart (`go.Funnel`) showing a hiring pipeline: Applied → Screened → Interviewed → Offered → Hired.
3. Create an animated scatter (`px.scatter` with `animation_frame`) showing salary vs score by year (generate 3 years of data).

## Mini Challenge

Build a complete interactive analytics dashboard using `make_subplots` with 4 panels:
- Salary histogram per department (colored traces)
- Box plot: salary by department
- Scatter: tenure vs salary colored by score
- Bar: promotion rate per department

Add a unified title and `template='plotly_white'`.

## Best Practices

- Use `px` for quick interactive exploration; switch to `go` for fine-grained control.
- Always set `template='plotly_white'` or `'ggplot2'` for publication-ready output.
- Use `fig.write_html('report.html')` to share interactive charts without Jupyter.
- Limit trace count in scatter plots (sample large datasets to < 10k points).

## Common Mistakes

- Showing Plotly figures in non-notebook environments without `.write_html`.
- Using `plt.show()` style with Plotly — call `.show()` on the figure object.
- 3D charts look impressive but are rarely clearer than 2D alternatives.

## Summary

- `go.Figure` + `make_subplots` = full dashboard control.
- Export with `fig.write_html`, `fig.write_image` (requires kaleido).
- Plotly + Dash = full Python web dashboard with callbacks and interactivity.

---


### Exercise and Challenge Solutions — Section 10


In [ ]:
try:
    import plotly.graph_objects as go

    # Funnel chart
    stages = ['Applied', 'Screened', 'Interviewed', 'Offered', 'Hired']
    counts = [500, 250, 100, 40, 25]
    fig = go.Figure(go.Funnel(
        y=stages, x=counts,
        textinfo='value+percent initial',
        marker=dict(color=['#636efa','#ef553b','#00cc96','#ab63fa','#ffa15a'])
    ))
    fig.update_layout(title='Hiring Funnel', template='plotly_white')
    fig.show()
except ImportError:
    print('Plotly not installed.')

In [ ]:
try:
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

    depts = df['department'].unique()
    colors_map = dict(zip(depts, px.colors.qualitative.Set2[:len(depts)]))

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Salary Histograms', 'Box: Salary by Dept',
                        'Tenure vs Salary (color=score)', 'Promotion Rate per Dept'),
        vertical_spacing=0.15, horizontal_spacing=0.12
    )

    # (1,1) Histograms per dept
    for dept in depts:
        sub = df[df['department']==dept]
        fig.add_trace(go.Histogram(x=sub['salary'], name=dept, opacity=0.6,
                                   marker_color=colors_map[dept], nbinsx=15), row=1, col=1)

    # (1,2) Box plots
    for dept in depts:
        sub = df[df['department']==dept]
        fig.add_trace(go.Box(y=sub['salary'], name=dept, marker_color=colors_map[dept],
                             showlegend=False), row=1, col=2)

    # (2,1) Scatter tenure vs salary colored by score
    fig.add_trace(go.Scatter(
        x=df['tenure'], y=df['salary'], mode='markers',
        marker=dict(color=df['score'], colorscale='Viridis', size=6, opacity=0.7,
                    colorbar=dict(title='Score', x=0.45)),
        name='Scatter', showlegend=False
    ), row=2, col=1)

    # (2,2) Promotion rate bar
    promo_rate = df.groupby('department')['promoted'].mean().sort_values(ascending=False)
    fig.add_trace(go.Bar(x=promo_rate.index, y=promo_rate.values,
                         marker_color=[colors_map[d] for d in promo_rate.index],
                         showlegend=False, name='Promo Rate'), row=2, col=2)

    fig.update_layout(
        height=700, title_text='Interactive Analytics Dashboard',
        template='plotly_white', barmode='overlay'
    )
    fig.show()
except ImportError:
    print('Plotly not installed.')

# Course Summary

## Matplotlib

| Section | Key Skills |
|---|---|
| 1. Object Model | `fig, ax = plt.subplots()`, artists, styles |
| 2. Line/Scatter/Bar | Trend, relationship, comparison charts |
| 3. Distributions | Histogram, box, violin — inspect before modeling |
| 4. Subplots | Grids, `subplot_mosaic`, suptitle |
| 5. Annotations | `annotate`, `axvspan`, formatters |

## Seaborn

| Section | Key Skills |
|---|---|
| 6. Statistical Plots | scatterplot, barplot, violinplot, catplot |
| 7. Heatmap/Pairplot | Correlation matrix, pair plots, joint plots |
| 8. Regression | regplot, lmplot, FacetGrid |

## Plotly

| Section | Key Skills |
|---|---|
| 9. Express | Interactive scatter, bar, time series, treemap |
| 10. Graph Objects | make_subplots, go traces, candlestick, funnel |

## Choosing the Right Tool

| Situation | Tool |
|---|---|
| Publication / paper | Matplotlib (full control) |
| Quick EDA with stats | Seaborn |
| Interactive dashboard | Plotly / Dash |
| Web embedding | Plotly (HTML output) |
| Presentation slides | Seaborn or Plotly |

## Next Steps

- **`statistics_ds_course.ipynb`** — EDA, feature engineering, statistical tests
- **`machine_learning_course.ipynb`** — Scikit-learn

---
